# 04 — Embeddings & Vector Stores (Meaning as Coordinates)

**By the end of this notebook, you can convert text into numeric embeddings, measure cosine similarity, visualize 2D meaning maps with PCA, and persist a vector store.**

### Learning Objectives
- Understand what an embedding vector is using a map-coordinates analogy.
- Measure semantic closeness with dot-product / cosine similarity and inspect 2D PCA coordinates.
- Index real chunks in `InMemoryVectorStore`, dump to JSON on disk, and reload with identical results.

### Prerequisites
- Notebooks 00–03

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: Words to Coordinates on a Meaning Map

Computers cannot compare the *meaning* of two English sentences with `==`. Instead, an **Embedding Model** acts like a master cartographer that places every phrase onto a multi-dimensional map (for example, 384 coordinates).

- Texts with related meanings (e.g., *"Enterprise pricing plan"* and *"USD 4,200 monthly subscription"*) land close together.
- Unrelated texts (*"chocolate cake recipe"*) land far away!
- **Cosine Similarity** measures how tightly two meaning vectors point in the same direction (`1.0` = same meaning, `0.0` = unrelated).
- A **Vector Store** acts as an indexed phonebook of coordinates so queries find top matches in milliseconds.

```
"Pets on Tuesday"  --> [ 0.21, -0.48, ..., 0.77 ]  \
  "Dogs in Zurich"   --> [ 0.19, -0.44, ..., 0.75 ]  --> High Cosine Similarity (~0.7+)
  "ROS2 Linux boot"  --> [-0.62,  0.51, ..., 0.04 ]  --> Low Cosine Similarity  (~0.1)
```

## 3. Step-by-Step Build

### Step 1: Embed Four Short Sentences and Inspect Vector Dimensions
Let's embed four short sentences using our local embeddings helper and check vector shape and determinism.

In [2]:
import numpy as np
phrases = [
    "What is the monthly cost of the Enterprise tier?",
    "Enterprise Plan pricing is $4,200 per month with 200 nodes.",
    "Dogs are permitted in the Zurich hub on Tuesdays and Thursdays.",
    "Freshly baked chocolate chip cookies require butter and sugar.",
]
vecs = embeddings.embed_documents(phrases)
print(f"Embedded {len(vecs)} texts -> each vector has {len(vecs[0])} coordinates.")
        

Embedded 4 texts -> each vector has 384 coordinates.


> **What just happened?**
> Every text—whether 5 words or a paragraph—maps to the exact same 384-dimensional vector space.

### Step 2: Compute Cosine Similarity Between Query and Candidates
Since normalized vectors have length 1.0, dot product equals cosine similarity. Let's compare `phrases[0]` against all four candidates.

> **Teaching Note (E2):** This manual cosine similarity computation is implemented from scratch for educational intuition; the canonical library/utility equivalent is provided in utils.models.cosine_similarity and scikit-learn (cosine_similarity).

In [3]:
q_vec = np.array(embeddings.embed_query(phrases[0]))
for p, v in zip(phrases, vecs):
    sim = float(np.dot(q_vec, np.array(v)) / (np.linalg.norm(q_vec) * np.linalg.norm(v) + 1e-9))
    print(f"sim={sim:+.3f} | {p}")
        

sim=+1.000 | What is the monthly cost of the Enterprise tier?
sim=+0.729 | Enterprise Plan pricing is $4,200 per month with 200 nodes.
sim=+0.003 | Dogs are permitted in the Zurich hub on Tuesdays and Thursdays.
sim=-0.027 | Freshly baked chocolate chip cookies require butter and sugar.


> **What just happened?**
> Notice how pricing question (`phrases[0]`) aligns highest with pricing answer (`phrases[1]`) and far lower with cookies (`phrases[3]`).

### Step 3: Compress 384 Dimensions Down to 2D with PCA for Visual Intuition
Humans cannot visualize 384 axes, so let's project our 4 vectors to 2D `(x, y)` coordinates using scikit-learn's `PCA`.

In [4]:
from sklearn.decomposition import PCA
coords_2d = PCA(n_components=2).fit_transform(np.array(vecs))
for p, (x, y) in zip(phrases, coords_2d):
    print(f"({x:+.3f}, {y:+.3f}) <- {p[:48]}")
        

(-0.579, -0.105) <- What is the monthly cost of the Enterprise tier?
(-0.608, -0.014) <- Enterprise Plan pricing is $4,200 per month with
(+0.492, +0.748) <- Dogs are permitted in the Zurich hub on Tuesdays
(+0.695, -0.629) <- Freshly baked chocolate chip cookies require but


> **What just happened?**
> PCA gives you concrete x/y coordinates illustrating how related ideas cluster together on the 2D plane.

### Step 4: Build, Query, Save, and Reload an `InMemoryVectorStore`
Let's index the real Novasphere chunks into `InMemoryVectorStore`, persist to JSON with `.dump()`, reload cleanly with `.load()`, and verify identical answers.

In [5]:
import tempfile
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from utils.helpers import load_corpus_documents

chunks = RecursiveCharacterTextSplitter(chunk_size=360, chunk_overlap=40).split_documents(load_corpus_documents(data_dir))
store = InMemoryVectorStore.from_documents(chunks, embeddings)
with tempfile.TemporaryDirectory() as tmp:
    dump_path = str(Path(tmp) / "store.json")
    store.dump(dump_path)
    reloaded = InMemoryVectorStore.load(dump_path, embeddings)
print("Reloaded match:", reloaded.similarity_search("ZeroGravity timeout threshold", k=1)[0].metadata)
        

Reloaded match: {'source': 'product_faq.md', 'doc_type': 'faq', 'access_level': 'public'}


> **What just happened?**
> Persisting your index with `.dump()` and `.load()` avoids re-embedding your entire library every time your app restarts.

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Test empty input safety** — *Hint:* Verify `embeddings.embed_query('')` returns a clean list of floats without throwing an error.
2. **Search k larger than corpus** — *Hint:* Query a 2-doc store with `k=10` and verify it returns all available docs cleanly.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [6]:
empty_vec = embeddings.embed_query("")
assert len(empty_vec) == 384
tiny_store = InMemoryVectorStore.from_texts(["Alpha telemetry", "Beta safety"], embeddings)
print("Oversized k returned:", len(tiny_store.similarity_search("telemetry", k=10)), "docs cleanly")
        

Oversized k returned: 2 docs cleanly


## 5. Common Mistakes
- Mixing two different embedding models (e.g., indexing with `MiniLM` and querying with `OpenAI`) — vectors live in completely incompatible spaces!
- Forgetting L2 normalization when writing manual dot-product similarity formulas.
- Expecting exact string equality (`sim == 1.0`) when searching paraphrased queries.

## 6. Recap
- Embeddings encode semantic meaning as fixed-length numeric arrays (vectors).
- Cosine similarity ranks closeness by angle between vectors rather than exact keyword overlap alone.
- `InMemoryVectorStore` lets you build, query, dump, and reload vector indexes natively in pure Python.

## 7. Next Notebook
Next up (`05_retrieval_and_similarity.ipynb`): Master Top-k tuning, similarity score cutoffs, and MMR diversity search so you don't retrieve 3 copies of the same paragraph!

## 8. Video Chapter Outline
1. **The Meaning Map Intuition** — How coordinates and angles encode semantic similarity.
2. **Embedding Real Sentences** — Vector dimensions and shapes in practice.
3. **Visualizing Meaning with PCA** — Compressing 384D vectors to intuitive 2D points.
4. **Vector Store Persistence** — Indexing, dumping JSON to disk, and reloading reliably.
5. **Recap & Next Steps** — Avoiding mismatched embedding spaces.